# 01 - Export ProLLaMA and Mol-LLaMA to ONNX 🚀

Run this notebook in a Colab TPU runtime with a high-memory host. Export runs on the TPU VM CPU; TPU cores are not used.

The notebook exports INT8 encoder graphs, checks PyTorch/ONNX parity, and uploads each encoder to a separate Hugging Face model repository.

In [1]:
WK_DIR = "/content/"

In [2]:
%cd {WK_DIR}
%cd Protein-Ligand-Affinity-Prediction-Using-LLM
!git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
%pip install -U pip setuptools wheel ninja packaging
%pip install -e ".[export]"
!mkdir -p external
%cd external/Mol-LLaMA
!git clone --depth 1 https://github.com/DongkiKim95/Mol-LLaMA external/Mol-LLaMA
!sed -e 's/rdkit-pypi/rdkit/g' -e '/^torch==/d' -e '/^torchvision==/d' -e '/^torchaudio==/d' -e '/^torch-geometric/d' -e '/^torch_scatter/d' -e '/^torch-scatter/d' -e '/^peft==/d' external/Mol-LLaMA/requirements.txt > {WK_DIR}/mol_llama_requirements.txt
%pip install -r {WK_DIR}/mol_llama_requirements.txt
%pip install "peft>=0.17,<1" openbabel-wheel ogb torch-geometric

/content
[Errno 2] No such file or directory: 'Protein-Ligand-Affinity-Prediction-Using-LLM'
/content
Cloning into 'Protein-Ligand-Affinity-Prediction-Using-LLM'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 116 (delta 40), reused 106 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 227.20 KiB | 3.79 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/Protein-Ligand-Affinity-Prediction-Using-LLM
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 15.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninsta

Obtaining file:///content/Protein-Ligand-Affinity-Prediction-Using-LLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 141.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 174.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 49.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 126.0 MB/s  0:00:00
  Building editable for protein-compound-affinity (pyproject.toml) ... done
  Created wheel for protein-compound-affinity: filename=protein_compound_affinity-0.1.0-0.editable-py3-none-any.whl size=6271 sha256=dd3a623f8fd79776fb6eeee6cfc0cb88efb0acd9e6a3591dc64e6ec3443c6f6d
  Stored in directory: /tmp/pip-ephem-wheel-cache-stypx5oh/wheels/2d/15/32/8b3ce02b265ec999071a21a74c05e75407b212b465bb4bb4e8
Successfully

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 142.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 90.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ogb]


In [3]:
import subprocess, sys, torch
version = '.'.join(torch.__version__.split('+')[0].split('.')[:2]) + '.0'
tag = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
url = f'https://data.pyg.org/whl/torch-{version}+{tag}.html'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', url])
print('Restart the Colab session after this cell, then continue.')

Restart the Colab session after this cell, then continue.


## Configuration

In [9]:
sys.path.insert(0,)

['/content',
 '/env/python',
 '/usr/lib/python311.zip',
 '/usr/lib/python3.11',
 '/usr/lib/python3.11/lib-dynload',
 '',
 '/usr/local/lib/python3.11/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.11/dist-packages/IPython/extensions',
 '/root/.ipython',
 '/usr/local/lib/python3.11/dist-packages/setuptools/_vendor']

In [10]:
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
from google.colab import userdata
from huggingface_hub import HfApi, login
from pathlib import Path
import os, subprocess
sys.path.insert(0,f"{WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM/src")
HF_USER = userdata.get('HF_USER')
PRO_REPO = f'{HF_USER}/prollama-affinity-onnx'
MOL_REPO = f'{HF_USER}/mol-llama-affinity-onnx'
ROOT = Path(f'{WK_DIR}/protein_affinity/onnx')
PRO_ONNX = ROOT / 'prollama'
MOL_ONNX = ROOT / 'mol_llama'
PRO_ONNX.mkdir(parents=True, exist_ok=True)
MOL_ONNX.mkdir(parents=True, exist_ok=True)
token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = token
login(token=token)

/content/Protein-Ligand-Affinity-Prediction-Using-LLM


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Export ProLLaMA

In [5]:
if not (PRO_ONNX / 'prollama_encoder.onnx').exists():
    subprocess.run([
        'affinity-export-protein-onnx',
        '--model-id', 'GreatCaptainNemo/ProLLaMA',
        '--output', str(PRO_ONNX),
        '--dtype', 'float32',
        '--sequence-length', '512',
        '--skip-parity-check',
        # '--quantize',
    ], check=True)
else:
    print('ProLLaMA export already exists.')

## Export Mol-LLaMA molecular encoder

In [6]:
if not (MOL_ONNX / 'mol_llama_encoder.onnx').exists():
    subprocess.run([
        'affinity-export-mol-onnx',
        '--official-repo', 'external/Mol-LLaMA',
        '--model-id', 'DongkiKim/Mol-Llama-3.1-8B-Instruct',
        '--output', str(MOL_ONNX),
        # '--quantize',
    ], check=True)
else:
    print('Mol-LLaMA export already exists.')

## ONNX smoke test

In [11]:
RUN_SMOKE_TEST = False  # Restart the Kaggle session before changing this to True.
if RUN_SMOKE_TEST:
    from affinity.onnx_embeddings import ProLLaMAOnnxEmbedder, MolLLaMAOnnxEmbedder
    protein = ProLLaMAOnnxEmbedder(PRO_ONNX).encode(['ACDEFGHIKLMNPQRSTVWY'])
    molecule = MolLLaMAOnnxEmbedder(MOL_ONNX).encode(['CC(=O)O'])
    print('Protein embedding:', protein.shape)
    print('Molecule embedding:', molecule.shape)
else:
    print('Smoke test skipped. Upload the artifacts, restart, then run validation.')

Protein embedding: (1, 4096)
Molecule embedding: (1, 768)


## Upload both encoder repositories

In [ ]:
UPLOAD = True
if UPLOAD:
    api = HfApi(token=token)
    for repo_id, folder in [(PRO_REPO, PRO_ONNX), (MOL_REPO, MOL_ONNX)]:
        api.create_repo(repo_id, repo_type='model', exist_ok=True)
        api.upload_large_folder(repo_id=repo_id, repo_type='model', folder_path=str(folder))
        print('Uploaded:', repo_id)

Recovering from metadata files:   0%|          | 0/297 [00:00<?, ?it/s]




---------- 2026-06-13 20:58:34 (0:00:00) ----------
Files:   hashed 3/297 (49.2K/26.4G) | pre-uploaded: 0/0 (0.0/26.4G) (+297 unsure) | committed: 0/297 (0.0/26.4G) | ignored: 0
Workers: hashing: 21 | get upload mode: 1 | pre-uploading: 0 | committing: 0 | waiting: 0
---------------------------------------------------


Uploading...:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

...

Uploading...:   0%|          | 0.00/180M [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/180M [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/180M [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/180M [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/67.1M [00:00<?, ?B/s]